# SML Project

## Data Reading and Preprocessing

### Flight data

In [6]:
import pandas as pd
import numpy as np
import os

# ==================== 配置 ====================
DATA_PATH = "/Users/clintliu/Desktop/group asmt/data/raw/"
INPUT_FILE = "flight_data_2024.csv"

# 我们只需要的列(预测时刻已知的特征 + label 来源 + 过滤用)
KEEP_COLUMNS = [
    # 时间
    'year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
    # 航司
    'op_unique_carrier', 'op_carrier_fl_num',
    # 机场
    'origin', 'origin_city_name', 'origin_state_nm',
    'dest', 'dest_city_name', 'dest_state_nm',
    # 计划时间(可作为特征)
    'crs_dep_time', 'crs_arr_time', 'crs_elapsed_time',
    'distance',
    # Label 来源(只用于算 label,不作为特征!)
    'arr_delay',
    # 过滤用
    'cancelled', 'diverted',
]

# ==================== 读取 ====================
print("=" * 60)
print("Block 1: 读取数据")
print("=" * 60)

filepath = os.path.join(DATA_PATH, INPUT_FILE)
print(f"读取文件: {filepath}")
print(f"文件大小: {os.path.getsize(filepath) / 1e6:.1f} MB")
print(f"只读取 {len(KEEP_COLUMNS)} 列以节省内存...")

df_raw = pd.read_csv(filepath, usecols=KEEP_COLUMNS, low_memory=False)

print(f"\n读取完成!")
print(f"  总行数: {len(df_raw):,}")
print(f"  列数: {len(df_raw.columns)}")
print(f"  内存占用: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\n日期范围: {df_raw['fl_date'].min()} 到 {df_raw['fl_date'].max()}")

# 看一下每月的航班数,确认数据覆盖全年
print("\n每月航班数:")
print(df_raw['month'].value_counts().sort_index())

Block 1: 读取数据
读取文件: /Users/clintliu/Desktop/group asmt/data/raw/flight_data_2024.csv
文件大小: 1309.0 MB
只读取 20 列以节省内存...

读取完成!
  总行数: 7,079,081
  列数: 20
  内存占用: 4336.1 MB

日期范围: 2024-01-01 到 2024-12-31

每月航班数:
month
1     547271
2     519221
3     591767
4     582205
5     609743
6     611132
7     634613
8     619025
9     582622
10    615497
11    575404
12    590581
Name: count, dtype: int64


In [7]:
# ==================== Block 2: 清洗 ====================
print("=" * 60)
print("Block 2: 清洗")
print("=" * 60)

df_clean = df_raw.copy()
print(f"清洗前: {len(df_clean):,} 行\n")

# ---------- 2.1 排除取消和改道的航班 ----------
# 这些航班没有 arr_delay,无法构造 label
before = len(df_clean)
n_cancelled = (df_clean['cancelled'] == 1).sum()
n_diverted = (df_clean['diverted'] == 1).sum()
df_clean = df_clean[(df_clean['cancelled'] == 0) & (df_clean['diverted'] == 0)].copy()
print(f"[2.1] 排除取消({n_cancelled:,})和改道({n_diverted:,})的航班")
print(f"      → 剩余: {len(df_clean):,} (删除 {before - len(df_clean):,})\n")

# ---------- 2.2 排除 arr_delay 缺失 ----------
# 理论上排除取消/改道后应该没有,以防万一
before = len(df_clean)
df_clean = df_clean.dropna(subset=['arr_delay'])
print(f"[2.2] 排除 arr_delay 缺失")
print(f"      → 剩余: {len(df_clean):,} (删除 {before - len(df_clean):,})\n")

# ---------- 2.3 构造 label: arr_del15 ----------
# 美国 DOT 官方标准:到达延误 ≥ 15 分钟视为 delayed
df_clean['arr_del15'] = (df_clean['arr_delay'] >= 15).astype(int)
delay_rate = df_clean['arr_del15'].mean() * 100
print(f"[2.3] 构造 label: arr_del15 = (arr_delay >= 15)")
print(f"      延误率: {delay_rate:.2f}%")
print(f"      准点: {(df_clean['arr_del15']==0).sum():,}")
print(f"      延误: {(df_clean['arr_del15']==1).sum():,}\n")

# ---------- 2.4 检查关键字段缺失 ----------
# 这些字段都是我们后续要用的特征
key_features = ['op_unique_carrier', 'origin', 'dest',
                'crs_dep_time', 'crs_arr_time', 'distance',
                'day_of_week', 'month']
print(f"[2.4] 关键特征字段缺失情况:")
missing = df_clean[key_features].isnull().sum()
missing_filtered = missing[missing > 0]
if len(missing_filtered) == 0:
    print("      ✅ 所有关键字段无缺失")
else:
    print(missing_filtered)
    # 把有缺失的行也删掉
    before = len(df_clean)
    df_clean = df_clean.dropna(subset=key_features)
    print(f"      删除缺失行后: {len(df_clean):,} (删除 {before - len(df_clean):,})")

# ---------- 2.5 排除可能的数据异常 ----------
# distance 必须为正
before = len(df_clean)
df_clean = df_clean[df_clean['distance'] > 0]
# crs_dep_time 在 [0, 2400) 之间(BTS 用 HHMM 格式)
df_clean = df_clean[(df_clean['crs_dep_time'] >= 0) & (df_clean['crs_dep_time'] < 2400)]
print(f"\n[2.5] 排除异常值(distance<=0 或 crs_dep_time 越界)")
print(f"      → 剩余: {len(df_clean):,} (删除 {before - len(df_clean):,})")

# ---------- 2.6 确保时间字段正确 ----------
df_clean['fl_date'] = pd.to_datetime(df_clean['fl_date'])

# ---------- 总结 ----------
print(f"\n{'='*60}")
print(f"清洗后总数: {len(df_clean):,}")
print(f"延误率: {df_clean['arr_del15'].mean()*100:.2f}%")
print(f"覆盖天数: {df_clean['fl_date'].dt.date.nunique()}")
print(f"航司数: {df_clean['op_unique_carrier'].nunique()}")
print(f"起飞机场数: {df_clean['origin'].nunique()}")
print(f"到达机场数: {df_clean['dest'].nunique()}")
print(f"{'='*60}")

Block 2: 清洗
清洗前: 7,079,081 行

[2.1] 排除取消(96,315)和改道(17,499)的航班
      → 剩余: 6,965,267 (删除 113,814)

[2.2] 排除 arr_delay 缺失
      → 剩余: 6,965,267 (删除 0)

[2.3] 构造 label: arr_del15 = (arr_delay >= 15)
      延误率: 20.82%
      准点: 5,515,295
      延误: 1,449,972

[2.4] 关键特征字段缺失情况:
      ✅ 所有关键字段无缺失

[2.5] 排除异常值(distance<=0 或 crs_dep_time 越界)
      → 剩余: 6,965,266 (删除 1)

清洗后总数: 6,965,266
延误率: 20.82%
覆盖天数: 366
航司数: 15
起飞机场数: 348
到达机场数: 348


In [8]:
# ==================== Block 3: 采样 ====================
print("=" * 60)
print("Block 3: 按日期分层采样")
print("=" * 60)

# ---------- 配置 ----------
SAMPLES_PER_DAY = 400          # 每天采 400 条 → 全年约 14.6 万
RANDOM_SEED = 42               # 固定 seed,保证可复现
OUTPUT_PATH = "/Users/clintliu/Desktop/group asmt/data/flights_2024_sampled.csv"

# ---------- 3.1 按日期分层采样 ----------
print(f"采样策略: 每天 {SAMPLES_PER_DAY} 条,random_seed={RANDOM_SEED}")

np.random.seed(RANDOM_SEED)
sampled_dfs = []
day_stats = []

for date, group in df_clean.groupby('fl_date'):
    n = min(SAMPLES_PER_DAY, len(group))
    day_stats.append((date, len(group), n))
    sampled = group.sample(n=n, random_state=RANDOM_SEED)
    sampled_dfs.append(sampled)

df_sampled = pd.concat(sampled_dfs, ignore_index=True)

print(f"\n采样结果:")
print(f"  总样本数: {len(df_sampled):,}")
print(f"  覆盖天数: {df_sampled['fl_date'].dt.date.nunique()}")
print(f"  平均每天: {len(df_sampled) / df_sampled['fl_date'].dt.date.nunique():.1f}")

# 检查是否有某天数据不足
low_days = [(d, total, sampled) for d, total, sampled in day_stats
            if sampled < SAMPLES_PER_DAY]
if low_days:
    print(f"\n⚠️  有 {len(low_days)} 天的数据不足 {SAMPLES_PER_DAY} 条:")
    for d, total, sampled in low_days[:5]:
        print(f"     {d.date()}: 仅 {total} 条")

# ---------- 3.2 按时间排序 ----------
df_sampled = df_sampled.sort_values(
    by=['fl_date', 'crs_dep_time']
).reset_index(drop=True)
print(f"\n已按 (fl_date, crs_dep_time) 排序")

# ---------- 3.3 保存 ----------
df_sampled.to_csv(OUTPUT_PATH, index=False)
print(f"\n已保存到: {OUTPUT_PATH}")
print(f"文件大小: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")

# ---------- 3.4 采样后的统计 ----------
print(f"\n{'='*60}")
print("采样后统计:")
print(f"{'='*60}")
print(f"延误率(全年): {df_sampled['arr_del15'].mean()*100:.2f}%")
print(f"\n按月份分布:")
monthly = df_sampled.groupby(df_sampled['fl_date'].dt.month).agg(
    samples=('arr_del15', 'size'),
    delay_rate=('arr_del15', lambda x: f"{x.mean()*100:.1f}%")
)
print(monthly)

Block 3: 按日期分层采样
采样策略: 每天 400 条,random_seed=42

采样结果:
  总样本数: 146,400
  覆盖天数: 366
  平均每天: 400.0

已按 (fl_date, crs_dep_time) 排序

已保存到: /Users/clintliu/Desktop/group asmt/data/flights_2024_sampled.csv
文件大小: 18.2 MB

采样后统计:
延误率(全年): 20.80%

按月份分布:
         samples delay_rate
fl_date                    
1          12400      24.4%
2          11600      16.0%
3          12400      20.9%
4          12000      19.4%
5          12400      26.5%
6          12000      25.7%
7          12400      29.3%
8          12400      24.5%
9          12000      15.0%
10         12400      12.5%
11         12000      14.0%
12         12400      20.8%


In [9]:
import pandas as pd

df = pd.read_csv("/Users/clintliu/Desktop/group asmt/data/flights_2024_sampled.csv")
df['fl_date'] = pd.to_datetime(df['fl_date'])

print("=" * 60)
print("最终数据健康检查")
print("=" * 60)

# 1. 总体信息
print(f"\n[1] 总体")
print(f"    总行数: {len(df):,}")
print(f"    列数: {len(df.columns)}")
print(f"    总延误率: {df['arr_del15'].mean()*100:.2f}%")

# 2. 各类别值的分布
print(f"\n[2] 类别字段")
print(f"    航司数量: {df['op_unique_carrier'].nunique()}")
print(f"    起飞机场数量: {df['origin'].nunique()}")
print(f"    到达机场数量: {df['dest'].nunique()}")
print(f"    起飞州数量: {df['origin_state_nm'].nunique()}")

# 3. 航司 top 5(看分布是否合理)
print(f"\n[3] 航司样本量 top 5:")
print(df['op_unique_carrier'].value_counts().head())

# 4. 机场 top 5
print(f"\n[4] 起飞机场样本量 top 5:")
print(df['origin'].value_counts().head())

# 5. 缺失检查
print(f"\n[5] 缺失值检查:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("    ✅ 无缺失")
else:
    print(missing[missing > 0])

# 6. 数值字段范围检查
print(f"\n[6] 数值字段范围:")
print(f"    distance: {df['distance'].min()} ~ {df['distance'].max()}")
print(f"    crs_dep_time: {df['crs_dep_time'].min()} ~ {df['crs_dep_time'].max()}")
print(f"    crs_elapsed_time: {df['crs_elapsed_time'].min()} ~ {df['crs_elapsed_time'].max()}")

# 7. 时间排序检查
print(f"\n[7] 时间排序检查:")
print(f"    首行日期: {df['fl_date'].iloc[0]}")
print(f"    末行日期: {df['fl_date'].iloc[-1]}")
is_sorted = (df['fl_date'].diff().dt.days.fillna(0) >= 0).all()
print(f"    日期是否递增: {'✅' if is_sorted else '❌'}")

print("\n" + "=" * 60)
print("✅ 数据准备完成!可以进入特征构造阶段了")
print("=" * 60)

最终数据健康检查

[1] 总体
    总行数: 146,400
    列数: 21
    总延误率: 20.80%

[2] 类别字段
    航司数量: 15
    起飞机场数量: 343
    到达机场数量: 346
    起飞州数量: 52

[3] 航司样本量 top 5:
op_unique_carrier
WN    28708
DL    21490
AA    19527
UA    15821
OO    15324
Name: count, dtype: int64

[4] 起飞机场样本量 top 5:
origin
ATL    7269
DFW    6348
DEN    6256
ORD    5791
CLT    4355
Name: count, dtype: int64

[5] 缺失值检查:
    ✅ 无缺失

[6] 数值字段范围:
    distance: 31.0 ~ 5095.0
    crs_dep_time: 1 ~ 2359
    crs_elapsed_time: 24.0 ~ 707.0

[7] 时间排序检查:
    首行日期: 2024-01-01 00:00:00
    末行日期: 2024-12-31 00:00:00
    日期是否递增: ✅

✅ 数据准备完成!可以进入特征构造阶段了


### Basic Dataset (S1)

In [10]:
import pandas as pd
import numpy as np

# ==================== Block 4: 基础特征构造 ====================
print("=" * 60)
print("Block 4: 基础特征构造")
print("=" * 60)

# 读取采样后的数据
INPUT_PATH = "/Users/clintliu/Desktop/group asmt/data/flights_2024_sampled.csv"
df = pd.read_csv(INPUT_PATH)
df['fl_date'] = pd.to_datetime(df['fl_date'])
print(f"加载: {len(df):,} 行")

# 保留一个"原始字段"副本,因为后续构造历史特征还需要它们
df_base = df.copy()

# ====================================================================
# 4.1 时间特征:从 fl_date 提取
# ====================================================================
print("\n[4.1] 时间特征(月份、星期、是否周末)")

# 月份 one-hot (12 个)
month_ohe = pd.get_dummies(df_base['month'], prefix='time_month').astype(int)

# 星期 one-hot (7 个)
dow_ohe = pd.get_dummies(df_base['day_of_week'], prefix='time_dow').astype(int)

# 是否周末(BTS 里 day_of_week: 1=Mon, 7=Sun)
df_base['time_is_weekend'] = df_base['day_of_week'].isin([6, 7]).astype(int)

print(f"  月份 one-hot: {month_ohe.shape[1]} 列")
print(f"  星期 one-hot: {dow_ohe.shape[1]} 列")
print(f"  是否周末: 1 列")

# ====================================================================
# 4.2 时间特征:从 crs_dep_time 提取
# ====================================================================
print("\n[4.2] 计划起飞时间特征(小时、时段)")

# crs_dep_time 是 HHMM 格式(1=00:01, 2359=23:59)
# 转换为小时数(0-23)
df_base['time_dep_hour'] = (df_base['crs_dep_time'] // 100).clip(0, 23)

# 计划起飞时间的连续值(小时 + 分钟/60),用于线性模型
df_base['time_dep_hour_continuous'] = (
    df_base['crs_dep_time'] // 100 +
    (df_base['crs_dep_time'] % 100) / 60
).clip(0, 24)

# 时段分桶(航空业常用的划分)
def get_time_bucket(hour):
    if 0 <= hour < 6:
        return 'late_night'    # 0-6 凌晨
    elif 6 <= hour < 9:
        return 'morning_peak'  # 6-9 早高峰
    elif 9 <= hour < 12:
        return 'morning'       # 9-12 上午
    elif 12 <= hour < 15:
        return 'afternoon'     # 12-15 下午
    elif 15 <= hour < 18:
        return 'evening_peak'  # 15-18 晚高峰
    elif 18 <= hour < 21:
        return 'evening'       # 18-21 傍晚
    else:
        return 'night'         # 21-24 夜间

df_base['time_bucket'] = df_base['time_dep_hour'].apply(get_time_bucket)
bucket_ohe = pd.get_dummies(df_base['time_bucket'], prefix='time_bucket').astype(int)

print(f"  起飞小时: 2 列(离散 + 连续)")
print(f"  时段分桶 one-hot: {bucket_ohe.shape[1]} 列")
print(f"  时段分布:")
print(df_base['time_bucket'].value_counts().to_string().replace('\n', '\n    '))

# ====================================================================
# 4.3 航司 one-hot
# ====================================================================
print("\n[4.3] 航司 one-hot")
airline_ohe = pd.get_dummies(df_base['op_unique_carrier'], prefix='airline').astype(int)
print(f"  航司 one-hot: {airline_ohe.shape[1]} 列")

# ====================================================================
# 4.4 机场 one-hot(top 50 + OTHER)
# ====================================================================
print("\n[4.4] 机场 one-hot(top 50 + OTHER)")

TOP_N_AIRPORTS = 50

# 起飞机场
top_origins = df_base['origin'].value_counts().head(TOP_N_AIRPORTS).index.tolist()
df_base['origin_grouped'] = df_base['origin'].where(
    df_base['origin'].isin(top_origins), 'OTHER'
)
origin_ohe = pd.get_dummies(df_base['origin_grouped'], prefix='origin').astype(int)

# 到达机场
top_dests = df_base['dest'].value_counts().head(TOP_N_AIRPORTS).index.tolist()
df_base['dest_grouped'] = df_base['dest'].where(
    df_base['dest'].isin(top_dests), 'OTHER'
)
dest_ohe = pd.get_dummies(df_base['dest_grouped'], prefix='dest').astype(int)

# 计算"被归入 OTHER 的样本占比"
origin_other_pct = (df_base['origin_grouped'] == 'OTHER').mean() * 100
dest_other_pct = (df_base['dest_grouped'] == 'OTHER').mean() * 100
print(f"  起飞机场 one-hot: {origin_ohe.shape[1]} 列(top {TOP_N_AIRPORTS} + OTHER 占 {origin_other_pct:.1f}%)")
print(f"  到达机场 one-hot: {dest_ohe.shape[1]} 列(top {TOP_N_AIRPORTS} + OTHER 占 {dest_other_pct:.1f}%)")

# ====================================================================
# 4.5 距离与飞行时长
# ====================================================================
print("\n[4.5] 距离与飞行时长")

# 数值特征(直接保留)
df_base['flight_distance'] = df_base['distance']
df_base['flight_crs_elapsed_time'] = df_base['crs_elapsed_time']

# 距离分桶(短途/中途/长途)
def get_distance_bucket(d):
    if d < 500:
        return 'short'      # < 500 英里
    elif d < 1500:
        return 'medium'     # 500-1500
    elif d < 2500:
        return 'long'       # 1500-2500
    else:
        return 'transcon'   # > 2500(跨大陆)

df_base['distance_bucket'] = df_base['distance'].apply(get_distance_bucket)
dist_ohe = pd.get_dummies(df_base['distance_bucket'], prefix='flight_dist').astype(int)

print(f"  距离原始值 + crs_elapsed_time: 2 列")
print(f"  距离分桶 one-hot: {dist_ohe.shape[1]} 列")
print(f"  距离分桶分布:")
print(df_base['distance_bucket'].value_counts().to_string().replace('\n', '\n    '))

# ====================================================================
# 4.6 组装特征矩阵
# ====================================================================
print("\n[4.6] 组装基础特征矩阵")

# 保留的原始字段(给后续历史特征构造用,不进入最终模型)
keep_meta = df_base[['fl_date', 'op_unique_carrier', 'origin', 'dest',
                     'crs_dep_time', 'arr_del15']].copy()

# 数值特征(单列)
numeric_features = df_base[[
    'time_dep_hour',
    'time_dep_hour_continuous',
    'time_is_weekend',
    'flight_distance',
    'flight_crs_elapsed_time',
]].copy()

# 拼接所有 one-hot
X_base = pd.concat([
    numeric_features,
    month_ohe,        # 12
    dow_ohe,          # 7
    bucket_ohe,       # 7
    airline_ohe,      # 15
    origin_ohe,       # 51
    dest_ohe,         # 51
    dist_ohe,         # 4
], axis=1)

# label 单独保留
y = df_base['arr_del15'].copy()

print(f"\n  基础特征矩阵 X_base shape: {X_base.shape}")
print(f"  label y shape: {y.shape}")
print(f"  延误率: {y.mean()*100:.2f}%")

# ====================================================================
# 4.7 检查 & 保存
# ====================================================================
print("\n[4.7] 检查与保存")

# 缺失检查
n_missing = X_base.isnull().sum().sum()
print(f"  缺失值: {n_missing}  {'✅' if n_missing == 0 else '⚠️'}")

# 类型检查
print(f"  特征类型分布: {X_base.dtypes.value_counts().to_dict()}")

# 特征前缀统计
prefix_counts = {}
for col in X_base.columns:
    prefix = col.split('_')[0]
    prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1
print(f"  按前缀分组:")
for prefix, count in sorted(prefix_counts.items()):
    print(f"    {prefix}_*: {count} 列")

# 保存
OUTPUT_DIR = "/Users/clintliu/Desktop/group asmt/data/"
X_base.to_csv(OUTPUT_DIR + "X_base.csv", index=False)
y.to_csv(OUTPUT_DIR + "y.csv", index=False)
keep_meta.to_csv(OUTPUT_DIR + "meta.csv", index=False)

print(f"\n  已保存:")
print(f"    {OUTPUT_DIR}X_base.csv  (基础特征矩阵)")
print(f"    {OUTPUT_DIR}y.csv  (label)")
print(f"    {OUTPUT_DIR}meta.csv  (原始字段,后续用)")

print("\n" + "=" * 60)
print(f"✅ Block 4 完成!基础特征: {X_base.shape[1]} 列")
print("   下一步: Block 5 构造历史统计特征(S3 场景的关键)")
print("=" * 60)

Block 4: 基础特征构造
加载: 146,400 行

[4.1] 时间特征(月份、星期、是否周末)
  月份 one-hot: 12 列
  星期 one-hot: 7 列
  是否周末: 1 列

[4.2] 计划起飞时间特征(小时、时段)
  起飞小时: 2 列(离散 + 连续)
  时段分桶 one-hot: 7 列
  时段分布:
time_bucket
    morning_peak    30452
    morning         26522
    evening_peak    26231
    afternoon       25974
    evening         23282
    night            9414
    late_night       4525

[4.3] 航司 one-hot
  航司 one-hot: 15 列

[4.4] 机场 one-hot(top 50 + OTHER)
  起飞机场 one-hot: 51 列(top 50 + OTHER 占 21.2%)
  到达机场 one-hot: 51 列(top 50 + OTHER 占 21.4%)

[4.5] 距离与飞行时长
  距离原始值 + crs_elapsed_time: 2 列
  距离分桶 one-hot: 4 列
  距离分桶分布:
distance_bucket
    medium      77171
    short       50528
    long        15677
    transcon     3024

[4.6] 组装基础特征矩阵

  基础特征矩阵 X_base shape: (146400, 152)
  label y shape: (146400,)
  延误率: 20.80%

[4.7] 检查与保存
  缺失值: 0  ✅
  特征类型分布: {dtype('int64'): 149, dtype('float64'): 3}
  按前缀分组:
    airline_*: 15 列
    dest_*: 51 列
    flight_*: 6 列
    origin_*: 51 列
    time_*: 29 列

  已保存:
    /Use